# Phase E: Data Provenance and Sanity Investigation

This notebook is a development-only diagnostic of the supplied CSV. It does not modify source data, train models, or inspect the final 100-row holdout for decisions.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
if not (project_root / 'src').exists():
    project_root = Path.cwd()
sys.path.insert(0, str(project_root))

from src.data_preprocessing import chronological_split, load_and_validate_data
from src.data_sanity import (
    consecutive_row_diagnostics, create_sanity_figures, electrical_consistency_diagnostics,
    temperature_delta_diagnostics, thermal_consistency_diagnostics,
)

data = load_and_validate_data(project_root / 'EV_Battery_Charging_TR_Dataset_with_Notes (1).csv')
development_data, final_test_data = chronological_split(data)
print('Development rows:', len(development_data))
print('Final holdout preserved:', len(final_test_data))

In [ ]:
delta, delta_bins = temperature_delta_diagnostics(development_data)
print(delta.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
delta_bins

In [ ]:
thermal = thermal_consistency_diagnostics(development_data)
print('Ordering violations:', thermal['ordering_invalid_count'])
print('Maximum midpoint error:', thermal['maximum_absolute_midpoint_error'])
thermal['spread'].describe()

In [ ]:
electrical = electrical_consistency_diagnostics(development_data)
print('Power correlation:', electrical['power_correlation'])
print('Maximum absolute power error:', electrical['power_error_kW'].abs().max())
electrical['pack_cell_ratio'].describe()

In [ ]:
consecutive = consecutive_row_diagnostics(development_data)
print('Transitions:', consecutive['transitions'])
print('Unique ChargerID/CellID pairs:', consecutive['unique_charger_cell_pairs'])
consecutive['lag_one_correlations'].sort_values()

In [ ]:
development_data.groupby('ChargingStage')['MaxTemp_C'].agg(['count', 'mean', 'std', 'min', 'max'])
development_data.groupby('EventFlag')['MaxTemp_C'].agg(['count', 'mean', 'std', 'min', 'max'])

In [ ]:
figure_paths = create_sanity_figures(development_data, project_root / 'results' / 'figures')
[str(path.relative_to(project_root)) for path in figure_paths]

## Scope boundary

A row can be arithmetically coherent without representing a physically continuous experiment. Provenance must be established from documentation or the data provider; it cannot be inferred as fact from plausible-looking values.